[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/stress/stress.ipynb)

# Stress in an Elastic Cube

Squeeze and shear a block of elastic material, and every face of it feels a force per area, its traction: partly pushing along the face's normal, partly shearing along the face. Which faces are sheared depends on how the block is cut. Cut along the principal frame, no face is sheared at all; turn the cut, and the normal and shear traction on a face trace a circle, Mohr's. This notebook builds the stress of an isotropic material from its strain, finds the principal frame, and turns the cube to draw that circle.

In tensor notation the stress reads as $\sigma_{ij} = \lambda\,\varepsilon_{kk}\delta_{ij} + 2\mu\,\varepsilon_{ij}$, and the traction on a face with unit normal $n$ as $t_i = \sigma_{ij} n_j$.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
import numpy as np
from IPython.display import Image, display

from numga import NumpyContext, stack
from numga.algebras import VGA3D
from examples.animation import save_animation
from examples.mechanics.stress import render
from examples.mechanics.stress.core import Views, corners, normals

ga = VGA3D
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
Vector = ga.gatype.vector()
Rotor = ga.gatype.rotor()
Strain = ga.gatype((Vector, Vector))                       # Vector <- Vector
Stress = ga.gatype((Vector, Vector))                       # Vector <- Vector

axes = mv.basis()                                          # [3] Vector
x, y, z = axes
# The material: the first Lamé parameter and the shear modulus.
LAME = 2.0
SHEAR_MODULUS = 1.0
# The strain: stretches along x, y and z, and a shear in the xy plane.
STRETCHES = np.array([0.05, -0.05, 0.08])
SHEAR = 0.30
# The frames of the turning view, over half a turn.
FRAMES = 60

## 1. Strain and stress

A strain is a map on vectors: how far each point moves, for where it sits. Here it is a stretch along each axis, written as a sum over the axes, plus a shear in the xy plane. The stress of an isotropic material is a map of the same kind: the strain's trace times the identity, weighted by the first Lamé parameter, plus twice the shear modulus times the strain. Given a face's normal it returns the traction across that face.

In [ ]:
strain = (axes * STRETCHES * (axes | Vector)).sum(axis=0) + SHEAR * (x * (y | Vector) + y * (x | Vector))   # [] Strain
stress = LAME * strain.trace() * Vector + 2 * SHEAR_MODULUS * strain            # [] Stress

## 2. The principal frame

`stress.eigh()` gives the principal stresses, from least to greatest, and their directions. The shear lies in the xy plane, so z stays principal and the turn in that plane taking x onto the first direction carries the axes onto the principal frame. The least and the greatest principal stress span Mohr's circle: its centre is their mean and its radius, half their difference, the largest shear any face feels.

In [ ]:
values, directions = stress.eigh()                                            # [3] Scalar, [3] Vector
principal = (1 + directions[0] * x).normalized()                               # [] Rotor
centre, radius = (values[0] + values[2]) / 2, (values[2] - values[0]).abs() / 2   # [] Scalar, [] Scalar

## 3. The cube from three frames

Seen from a turned frame the stress is `rotor << stress(rotor >> Vector)`. `views` takes a batch of frames and gives, for each, the deformed cube, the traction on each face split into its part along the normal, blue, and its shear, red, and the principal directions, dotted green. In the lab frame the faces are sheared; in the principal frame none is; an eighth of a turn on, the shear is the largest it gets. On Mohr's circles each frame is a diameter, joining the faces facing x and y.

In [ ]:
def views(strain, stress, directions, rotors) -> Views:
    """The cube seen from each frame: the strain and the stress as the frame sees them, the cube
    deformed, the tractions on its faces split along their normals and across, and the principal
    directions."""
    local_strain = rotors << strain(rotors >> Vector)                          # [frames] Strain
    local_stress = rotors << stress(rotors >> Vector)                          # [frames] Stress
    total = local_stress[:, None](normals)                                     # [frames, 6] Vector
    normal = normals * (normals | total)                                       # [frames, 6] Vector
    return Views(corners + local_strain[:, None](corners), 0.5 * (normals + local_strain[:, None](normals)),
                 normal, total - normal, rotors[:, None] << directions)


rotors = stack([mv.rotor(), principal, ((x ^ y) * (-np.pi / 8)).exp() * principal])   # [3] Rotor
seen = views(strain, stress, directions, rotors)
render.draw_frames(seen, values, ["lab frame", "principal frame", "greatest shear"]);

## 4. Turning the frame

Turning the frame half a turn in the shear plane from the principal frame, the cube squares up and shears again, and the diameter goes once around Mohr's circle.

In [ ]:
angles = np.linspace(0.0, np.pi, FRAMES, endpoint=False)
turning = views(strain, stress, directions, ((x ^ y) * (-angles / 2)).exp() * principal)   # [frames]
display(Image(filename=save_animation(render.animate_turning(turning, values), "stress_rotation", 50)))

In [ ]:
# checks
# In the principal frame no face is sheared, the normal tractions on the faces facing x, y and z are the
# principal stresses, and the turned axes their directions.
np.testing.assert_allclose(seen.shear[1].norm().to_array(), 0.0, atol=1e-14)
np.testing.assert_allclose(np.sort((seen.normal[1, :3] | axes).to_array()), values.to_array(), rtol=1e-12)
turned = principal >> axes                                                     # [3] Vector
np.testing.assert_allclose((stress(turned) ^ turned).norm().to_array(), 0.0, atol=1e-14)
# An eighth of a turn on, the face facing x feels the largest shear.
np.testing.assert_allclose(seen.shear[2, 0].norm().to_array(), radius.to_array(), rtol=1e-8)
# Turning, the normal and shear traction on the face facing x go around Mohr's circle.
normal, shear = turning.normal[:, 0] | x, turning.shear[:, 0] | y               # [frames] Scalar
np.testing.assert_allclose(((normal - centre).squared() + shear.squared()).to_array(), radius.squared().to_array(), rtol=1e-8)